In [1]:
import os
import ctypes
import numpy as np

# 🔧 Extend the search path for shared libraries
os.environ["LD_LIBRARY_PATH"] = (
    "/home/cpanourg/local/glpk/lib:"
    "/home/cpanourg/local/armadillo/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

# ✅ Load libVAQ.so now
vaq = ctypes.CDLL("/home/cpanourg/projects/2-hdvc/lib/VAQ/build/bitvecengine/libVAQ.so")

# Define function prototype
vaq.vaq_quantize.argtypes = [
    ctypes.POINTER(ctypes.c_float), ctypes.c_int, ctypes.c_int,
    ctypes.c_int, ctypes.c_int, ctypes.POINTER(ctypes.c_uint8)
]


OSError: libglpk.so.40: cannot open shared object file: No such file or directory

In [2]:
codes = vaq_quantize_numpy(np.random.randn(1000, 128).astype(np.float32))


NameError: name 'vaq_quantize_numpy' is not defined

In [2]:
import ctypes
import numpy as np

lib = ctypes.CDLL("/home/cpanourg/projects/2-hdvc/lib/VAQ/build/bitvecengine/libVAQ.so")

lib.vaq_train_and_encode.argtypes = [
    ctypes.POINTER(ctypes.c_float), ctypes.c_int, ctypes.c_int,
    ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int,
    ctypes.c_float, ctypes.POINTER(ctypes.c_uint16)
]

def vaq_quantize(data: np.ndarray,
                 total_bits=32, n_subspaces=8,
                 min_bits=1, max_bits=8,
                 var_threshold=0.95):
    data = np.ascontiguousarray(data.astype(np.float32))
    n, d = data.shape
    out_codes = np.zeros((n, n_subspaces), dtype=np.uint16)

    lib.vaq_train_and_encode(
        data.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        n, d,
        total_bits, n_subspaces,
        min_bits, max_bits,
        var_threshold,
        out_codes.ctypes.data_as(ctypes.POINTER(ctypes.c_uint16))
    )
    return out_codes


OSError: libglpk.so.40: cannot open shared object file: No such file or directory

In [30]:
import numpy as np 
import faiss
import sys
import time
import csv
import os
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from numba import jit, prange
import threading
from concurrent.futures import ThreadPoolExecutor
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm 
module_path = '/home/cpanourg/projects/2-hdvc/'

if module_path not in sys.path:
    sys.path.append(module_path)

from src.utils import read_fvecs, write_fvecs, read_ivecs
from src.utils import append_or_create_csv
from src.utils import compute_distance_tables_threaded, compute_distance_tables_vectorized, adc_distances_batch_numba\
                    , compute_all_distances_batch, adc_distances_all_optimized, lsq_alpha_from_codes \
                    , lsq_dot_tables_vectorized, lsq_distances_batch_numba
  
from datetime import datetime

# Current date & time
now = datetime.now()

# Format to a readable string
dt_str = now.strftime("%Y_%m_%d_%H_%M_%S")
print(dt_str)



2025_10_30_10_35_46


In [2]:
codes = read_fvecs('/data/cpanourg/2-hdvc/results/vaq/codes.fvecs')

Reading File - /data/cpanourg/2-hdvc/results/vaq/codes.fvecs:

ValueError: cannot reshape array of size 160004 into shape (10001)

In [ ]:
codes = np.fromfile("/data/cpanourg/2-hdvc/results/vaq/codes.fvecs", dtype=)


In [8]:
codes

array([            10000,                32, 31526146580873433, ...,
       43910581706883298, 56858855842316330, 11259450039992357])

Loaded codebook: (10000, 32) int16


In [21]:
codes

array([[217,  23, 221, ..., 107, 173,  32],
       [ 86,  83,  61, ..., 224,  53,  42],
       [217, 140,  39, ...,  23, 159,  37],
       ...,
       [ 76,  80,  19, ..., 161, 252,  85],
       [ 92,  52, 204, ...,  47,  46,  37],
       [ 57,  76,  47, ...,   0, 105,  40]], dtype=int16)

In [35]:



dataset_name = 'gist'

result_fp = f'/data/cpanourg/2-hdvc/results/vaq/{dataset_name}/{dt_str}'
os.makedirs(result_fp, exist_ok=True)

db_fp = '/data/cpanourg/2-hdvc/data/gist/gist_base.fvecs'
qr_fp = '/data/cpanourg/2-hdvc/data/gist/gist_query.fvecs'
gt_fp = '/data/cpanourg/2-hdvc/data/gist/gist_groundtruth.ivecs'
codes_fp = f'{result_fp}/codes.fvecs'

db = read_fvecs(db_fp)
qr = read_fvecs(qr_fp)

db_size = db.shape[0]
qr_size = qr.shape[0]
dim = db.shape[1]


Reading File - /data/cpanourg/2-hdvc/data/gist/gist_base.fvecs:(1000000, 960)
Reading File - /data/cpanourg/2-hdvc/data/gist/gist_query.fvecs:(1000, 960)


In [36]:
topk = 100

In [37]:
import subprocess
import os

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = (
    f"{os.path.expanduser('~')}/local/glpk/lib:"
    f"{os.path.expanduser('~')}/local/armadillo/lib:"
    f"{env.get('CONDA_PREFIX', '')}/lib:"
    f"{env.get('LD_LIBRARY_PATH', '')}"
)

cmd = f"""
/home/cpanourg/projects/2-hdvc/lib/VAQ/build/examples/demo_vaq \
  --dataset {db_fp} \
  --queries {qr_fp} \
  --file-format-ori fvecs \
  --timeseries-size {dim} \
  --dataset-size {db_size} \
  --queries-size {qr_size} \
  --result {result_fp}/answer_vaq_VAQ256m32min7max8var1,HEAP_refine100,200_sift_10K.csv \
  --groundtruth {gt_fp} \
  --groundtruth-format ivecs \
  --method VAQ256m32min7max8var1,HEAP \
  --k {topk} \
  --refine 100,200 \
  --save-enc {codes_fp}
"""

process = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Stream output line by line
for line in process.stdout:
    print(line, end="")

process.wait()
print(f"\n✅ Process finished with exit code {process.returncode}")


Arguments Passed:
	dataset = /data/cpanourg/2-hdvc/data/gist/gist_base.fvecs
	dataset-size = 1000000
	file-format-ori = fvecs
	groundtruth = /data/cpanourg/2-hdvc/data/gist/gist_groundtruth.ivecs
	groundtruth-format = ivecs
	hc-bitalloc = 
	k = 100
	kmeans-ver = 0
	learn-ratio = 0.05
	method = VAQ256m32min7max8var1,HEAP
	queries = /data/cpanourg/2-hdvc/data/gist/gist_query.fvecs
	queries-size = 1000
	refine = 100,200
	result = /data/cpanourg/2-hdvc/results/vaq/gist/2025_10_30_10_35_46/answer_vaq_VAQ256m32min7max8var1,HEAP_refine100,200_sift_10K.csv
	save = 
	save-enc = /data/cpanourg/2-hdvc/results/vaq/gist/2025_10_30_10_35_46/codes.fvecs
	timeseries-size = 960
	visit-cluster = 1
Preprocessing steps..

Read dataset
Training & encoding phase
Training the centroids
1000000 960


KeyboardInterrupt: 

In [ ]:
import numpy as np

with open(codes_fp, "rb") as f:
    nrows = np.fromfile(f, dtype=np.int64, count=1)[0]
    ncols = np.fromfile(f, dtype=np.int64, count=1)[0]
    codes = np.fromfile(f, dtype=np.int16, count=nrows * ncols).reshape(nrows, ncols)

print("Loaded codebook:", codes.shape, codes.dtype)

